# Diantace calculation and its diravatives

In [1]:
import sympy as sp
from sympy import symbols, integrate, expand

In [2]:
xbars = symbols('\\bar{x}_{1:13}', real=True)
xbar = sp.Matrix(xbars)
xbar.T

Matrix([[\bar{x}_{1}, \bar{x}_{2}, \bar{x}_{3}, \bar{x}_{4}, \bar{x}_{5}, \bar{x}_{6}, \bar{x}_{7}, \bar{x}_{8}, \bar{x}_{9}, \bar{x}_{10}, \bar{x}_{11}, \bar{x}_{12}]])

In [3]:
xs = symbols('x_{1:13}', real=True)
x = sp.Matrix(xs)
x.T

Matrix([[x_{1}, x_{2}, x_{3}, x_{4}, x_{5}, x_{6}, x_{7}, x_{8}, x_{9}, x_{10}, x_{11}, x_{12}]])

In [4]:
x1 = sp.Matrix(x[0:3, 0])
x2 = sp.Matrix(x[3:6, 0])
x3 = sp.Matrix(x[6:9, 0])
x4 = sp.Matrix(x[9:12, 0])
xbar1 = sp.Matrix(xbar[0:3, 0])
xbar2 = sp.Matrix(xbar[3:6, 0])
xbar3 = sp.Matrix(xbar[6:9, 0])
xbar4 = sp.Matrix(xbar[9:12, 0])

## Distance functions

In [5]:
dist_vv = (x1 - x2).norm()
dist_vv

sqrt((x_{1} - x_{4})**2 + (x_{2} - x_{5})**2 + (x_{3} - x_{6})**2)

In [6]:
dist_vv = (x1 - x2).norm()

dist_ve = ((x1 - x3).cross(x2 - x3)).norm() / (x1 - x2).norm()

p = x1
T1 = x2
T2 = x3
T3 = x4
dist_vf = ((p - T1) * ((T2 - T1).cross(T3 - T1)).norm()).norm()
# dist_vf = (x1 - x2) * ((x3 - x2).cross(x4 - x2).norm())

v11 = x1
v12 = x2
v21 = x3
v22 = x4
dist_ee = ((v11 - v21) * ((v12 - v11).cross(v22 - v21)).norm()).norm()

In [7]:
# Example values for xbar to calculate VE, VF, EE distances
example_values = {
    x[0]: 0, x[1]: 0, x[2]: 0,    # x1 / v11 / p
    x[3]: 1, x[4]: 0, x[5]: 0,    # x2 / v12 / T1
    x[6]: 0, x[7]: 1, x[8]: 0,    # x3 / v21 / T2
    x[9]: 0, x[10]: 0, x[11]: 1 # x4 / v22 / T3
}

vv_val = dist_vv.subs(example_values).evalf()
ve_val = dist_ve.subs(example_values).evalf()
vf_val = dist_vf.subs(example_values).evalf()
ee_val = dist_ee.subs(example_values).evalf()

print("VV distance:", vv_val)
print("VE distance:", ve_val)
print("VF distance:", vf_val)
print("EE distance:", ee_val)

VV distance: 1.00000000000000
VE distance: 1.00000000000000
VF distance: 1.73205080756888
EE distance: 1.41421356237310


## Contact Assembly for Affine-body: $\partial d / \partial q$

Since we use $q\in R^{12}$ for each rigid-body(affine body), but the contact energy is defined for vertices, so we need to transmit the influence of energy on $x$ to $q$ through the chain rule:

$$ \frac{\partial E}{\partial q_i} =  \frac{\partial E}{\partial d} \frac{\partial d}{\partial t} \frac{\partial t}{\partial q_i} = \frac{\partial E}{\partial d} \frac{\partial d}{\partial t} \sum_{j \in i}  \frac{\partial t}{\partial x_j} \frac{\partial x_j}{\partial q_i}  $$

where $j$ is the vertices with contact in body $i$, $d=|t|$ is the distance of primitives in collision pair, $t$ is the relative displacement vector.

If any element in collision pair (V/E/F in VF/EE pair) is from rigid body, then we can accumulate the influence of the primitives (Take F in VF pair as example):

$$
\begin{aligned}
\sum_i^F \frac{\partial t}{\partial q_i} 
&= \sum_i^F \frac{\partial t}{\partial x_i} \frac{\partial x_i}{\partial q_i}  \\
&= \sum_i^F \frac{\partial t}{\partial x_i} J_2 \\
&= (\sum_i^F \frac{\partial t}{\partial x_i}) J_2 \\
&= (\sum_i^F w_i I_{3 \times 3}) J_2 \\
&= (-I_{3 \times 3}) J_2 \\
&= - J_2
\end{aligned}
$$

So we have:

$$ \nabla E_{q_2} = \frac{\partial E}{\partial d} \frac{\partial d}{\partial t}  \nabla t_{q_2} = - J_2 n $$

> $ \frac{\partial d}{\partial t} = \frac{\partial |t|}{\partial t} =  \frac{t^T}{|t|} = n^T $, where $n$ is the direction between primitives.

And for hessian (second-order derivatives):


$$
\begin{aligned}
\sum_i \sum_j \frac{\partial^2 E}{\partial q_i \partial q_j} 
&= \sum_i \sum_j \frac{\partial x_i}{\partial q_i}  \frac{\partial^2 d}{\partial x_i x_j} \frac{\partial x_i}{\partial q_j}  \\
&= 
\sum_i \sum_j J_1^T \frac{\partial^2 E}{\partial d^2} \frac{\partial d}{\partial x_i} (\frac{\partial d}{\partial x_j})^T J_2 
+ 
\sum_i \sum_j J_1^T \frac{\partial E}{\partial d}  \frac{\partial^2 d}{\partial x_i x_j} J_2 \\
&= 
\frac{\partial^2 E}{\partial d^2} J_1^T  \sum_i \sum_j \frac{\partial d}{\partial x_i} (\frac{\partial d}{\partial x_j})^T J_2 
+ 
 \frac{\partial E}{\partial d} J_1^T  \sum_i \sum_j \frac{\partial^2 d}{\partial x_i x_j} J_2 \\
&= 
\frac{\partial^2 E}{\partial d^2} J_1^T  (\sum_i w_i \sum_j w_j) n n^T J_2 
+ 
 \frac{\partial E}{\partial d} J_1^T  \sum_i \sum_j \frac{\partial^2 d}{\partial x_i x_j} J_2 \\
\end{aligned}
$$

where $\sum_i w_i = \pm 1$ is the summury of weights.

If we use Gauss-Newton approximation for hessian calculation, then the $\frac{\partial^2 E}{\partial d^2} J_1^T  (\sum_i w_i \sum_j w_j) n n^T J_2$ is enough for calculation. 
<!-- 
As for $\sum_i \sum_j \frac{\partial^2 d}{\partial x_i x_j}$ term:

$$
\begin{aligned}
\sum_i \sum_j \frac{\partial^2 d}{\partial x_i x_j}
&= \sum_i ( \sum_j \frac{\partial}{\partial x_j}( \frac{\partial d}{\partial x_i}) ) \\
\end{aligned}
$$ -->



### Deviratives Calculation Examples

In [8]:
q = symbols('q1:13', real=True)
q = sp.Matrix(q)

p = sp.Matrix.zeros(3, 1)
A1 = sp.Matrix.zeros(3, 3)
A2 = sp.Matrix.zeros(3, 3)

p = q[0:3, 0]
A1[0, 0:3] = (q[3:6, 0]).T
A1[1, 0:3] = (q[6:9, 0]).T
A1[2, 0:3] = (q[9:12, 0]).T
A2[0:3, 0] = (q[3:6, 0])
A2[0:3, 1] = (q[6:9, 0])
A2[0:3, 2] = (q[9:12, 0])


J = dx1dq = (A1 * xbar1 + p).jacobian(q) # q should corespond with distribution of A
J

Matrix([
[1, 0, 0, \bar{x}_{1}, \bar{x}_{2}, \bar{x}_{3},           0,           0,           0,           0,           0,           0],
[0, 1, 0,           0,           0,           0, \bar{x}_{1}, \bar{x}_{2}, \bar{x}_{3},           0,           0,           0],
[0, 0, 1,           0,           0,           0,           0,           0,           0, \bar{x}_{1}, \bar{x}_{2}, \bar{x}_{3}]])

In [9]:
grad_vf = sp.Matrix([dist_vf]).jacobian(x).T
hess_vf = grad_vf.jacobian(x)
print(grad_vf.shape)
print(hess_vf.shape)

(12, 1)
(12, 12)


In [14]:
# x1 = A1 * xbar1 + p
# x2 = A1 * xbar2 + p
# x3 = A1 * xbar3 + p
# x4 = A1 * xbar4 + p

grad_vf_val = grad_vf.subs(example_values).evalf()
hess_vf_val = hess_vf.subs(example_values).evalf()
grad_vf_val
# print("VF distance hess:", hess_vf_val)


Matrix([
[ -1.73205080756888],
[                 0],
[                 0],
[  2.88675134594813],
[-0.577350269189626],
[-0.577350269189626],
[-0.577350269189626],
[  1.15470053837925],
[-0.577350269189626],
[-0.577350269189626],
[-0.577350269189626],
[  1.15470053837925]])

In [11]:
grad_q1 = J.T * grad_vf[0:3, 0]
grad_q2 = J.T * grad_vf[3:6, 0] + J.T * grad_vf[6:9, 0] + J.T * grad_vf[9:12, 0]
# = grad_q2 = J.T * (grad_vf[3:6, 0] + grad_vf[6:9, 0] + grad_vf[9:12, 0])
grad_q1_val = grad_q1.subs(example_values).evalf()
grad_q234_val = grad_q2.subs(example_values).evalf()
grad_q1_val + grad_q234_val

Matrix([
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0],
[0]])

In [12]:
hess_q1 = J.T * hess_vf[0:3, 0:3] * J
hess_q2 = J.T * (\
    hess_vf[3:6, 3:6] + hess_vf[3:6, 6:9] + hess_vf[3:6, 9:12] + \
        hess_vf[6:9, 3:6] + hess_vf[6:9, 6:9] + hess_vf[6:9, 9:12] + \
            hess_vf[9:12, 3:6] + hess_vf[9:12, 6:9] + hess_vf[9:12, 9:12]) * J
hess_q1_val = hess_q1.subs(example_values).evalf()
hess_q234_val = hess_q2.subs(example_values).evalf()
hess_q1_val - hess_q1_val

Matrix([
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [13]:
print(format("{:032b}".format(3 << 30)))

11000000000000000000000000000000
